# Data Loading

In [1]:
with open('data/small_vocab_en', 'r') as f:
    eng_sentences = f.read().split('\n')
    
with open('data/small_vocab_fr', 'r') as f:
    fre_sentences = f.read().split('\n')

print('Dataset Loaded')

Dataset Loaded


In [2]:
import collections
import numpy as np

print(eng_sentences[0:5],"...eng sentences...")
print(fre_sentences[0:5],"...french sentences..")
print(len(eng_sentences),"...length of english sentences...")
print(len(fre_sentences),"...length of french sentences....")

['new jersey is sometimes quiet during autumn , and it is snowy in april .', 'the united states is usually chilly during july , and it is usually freezing in november .', 'california is usually quiet during march , and it is usually hot in june .', 'the united states is sometimes mild during june , and it is cold in september .', 'your least liked fruit is the grape , but my least liked is the apple .'] ...eng sentences...
["new jersey est parfois calme pendant l' automne , et il est neigeux en avril .", 'les Ã©tats-unis est gÃ©nÃ©ralement froid en juillet , et il gÃ¨le habituellement en novembre .', 'california est gÃ©nÃ©ralement calme en mars , et il est gÃ©nÃ©ralement chaud en juin .', 'les Ã©tats-unis est parfois lÃ©gÃ¨re en juin , et il fait froid en septembre .', 'votre moins aimÃ© fruit est le raisin , mais mon moins aimÃ© est la pomme .'] ...french sentences..
137861 ...length of english sentences...
137861 ...length of french sentences....


# Pre Processing

In [3]:
import numpy as np
import io
import unicodedata
import re
from tqdm import tqdm

def unicode_to_ascii(s) :
  """
  Unicode to ascii conversion
  """
  return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def cleanhtml(raw_html) :
  """
  Function to clean html tags and numbers
  """
  cleanr = re.compile('<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
  cleantext = re.sub(cleanr, '', raw_html)
  return cleantext

def cleanString(incomingString):
    """
      Function to clean unwanted symbol from text
    """
    newstring = incomingString
    newstring = newstring.replace("!","")
    newstring = newstring.replace("@","")
    newstring = newstring.replace("#","")
    newstring = newstring.replace("$","")
    newstring = newstring.replace("%","")
    newstring = newstring.replace("^","")
    newstring = newstring.replace("&","and")
    newstring = newstring.replace("*","")
    newstring = newstring.replace("(","")
    newstring = newstring.replace(")","")
    newstring = newstring.replace("+","")
    newstring = newstring.replace("=","")
    newstring = newstring.replace("?","")
    newstring = newstring.replace("\'","")
    newstring = newstring.replace("\"","")
    newstring = newstring.replace("{","")
    newstring = newstring.replace("}","")
    newstring = newstring.replace("[","")
    newstring = newstring.replace("]","")
    newstring = newstring.replace("<","")
    newstring = newstring.replace(">","")
    newstring = newstring.replace("~","")
    newstring = newstring.replace("`","")
    newstring = newstring.replace(":","")
    newstring = newstring.replace(";","")
    newstring = newstring.replace("|","")
    newstring = newstring.replace("\\","")
    newstring = newstring.replace("/","")     
    return ' '.join(newstring.split())

def preprocess_string(data) :
  """
  This function calls other
  preprocessing function for
  cleaning data
  """
  data = unicode_to_ascii(data)
  #Remove html
  data = cleanhtml(data)
  #Remove unwanted symbols
  data = cleanString(data)
  return data


def start_preprocessing(data,lang):
    print("..started preprocessing..."+lang)
    preproc_data_list = []
    for val in tqdm(data):
        preproc_data=preprocess_string(val)
        preproc_data_list.append(preproc_data)
    return preproc_data_list    

In [4]:
eng_preproc_sentence = start_preprocessing(eng_sentences,"eng")
fre_preproc_senetence = start_preprocessing(fre_sentences,"french")

..started preprocessing...eng


100%|███████████████████████████████████████████████████████████████████████| 137861/137861 [00:05<00:00, 25390.51it/s]


..started preprocessing...french


100%|███████████████████████████████████████████████████████████████████████| 137861/137861 [00:06<00:00, 21450.99it/s]


# Translation english to french

In [5]:
# Putting the start and end words in the french sentances

eng_preproc_sentence = [x.lower() for x in eng_preproc_sentence]
fre_preproc_senetence = [x.lower() for x in fre_preproc_senetence]
#fre_preproc_senetence = ["start " + x + " end" for x in fre_preproc_senetence ]

In [6]:
eng_preproc_sentence = eng_preproc_sentence[0:500] 
fre_preproc_senetence = fre_preproc_senetence[0:500] 

In [7]:
# import sklearn
# print(sklearn.__version__)

In [8]:
from sklearn.model_selection import train_test_split
X=eng_preproc_sentence
Y=fre_preproc_senetence
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size = 0.1)
len(X_train),len(y_train), len(X_test), len(y_test)

C:\Users\utsav\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


(450, 450, 50, 50)

In [9]:
print(X_train[1], y_train[1])
print("......................................")
print(X_test[1],  y_test[1])

i think it is difficult to translate between spanish and portuguese . je pense quil est difficile de traduire entre l espagnol et le portugais .
......................................
china is never mild during april , and it is sometimes wet in spring . chine est jamais doux en avril , et il est parfois humide au printemps .


In [10]:
def Max_length(data):
  max_length_ = max([len(x.split(' ')) for x in data])
  return max_length_

#Training data
max_length_english = Max_length(X_train)
max_length_french = Max_length(y_train)

#Test data
max_length_english_test = Max_length(X_test)
max_length_french_test = Max_length(y_test)

print(max_length_english_test,"...max english length test..")
print(max_length_french_test,"....max french  length test..")

print(max_length_english,"...max english length train..")
print(max_length_french,"....max french  length train..")

17 ...max english length test..
18 ....max french  length test..
17 ...max english length train..
21 ....max french  length train..


# pytorch finetuning framework

In [11]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.model_selection import train_test_split
import time
import datetime

In [12]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'

In [13]:
# Initialize BERT tokenizer
# model_name = 'bert-base-uncased'
# tokenizer = BertTokenizer.from_pretrained(model_name)

############T5-SMALL####################################
model_name = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


##########BART###############################################

# model_name = 'facebook/bart-large'
# model = BartForConditionalGeneration.from_pretrained(model_name)
# tokenizer = BartTokenizer.from_pretrained(model_name)

############GPT-2#########################################
# tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# model = GPT2LMHeadModel.from_pretrained('gpt2')
# tokenizer.add_special_tokens({'pad_token': '[PAD]'})
# # Resize model embedding to account for the new token
# model.resize_token_embeddings(len(tokenizer))

# model_type="gpt"
# model = model.to(device)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
# Example translation dataset
class TranslationDataset(Dataset):
    def __init__(self, source_texts, target_texts, tokenizer):
        self.source_texts = source_texts
        self.target_texts = target_texts
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.source_texts)
     
    
    ###Seperate tokenization#######
#     def __getitem__(self, idx):
#         source_text = self.source_texts[idx]
#         target_text = self.target_texts[idx]
#         source_tokens = self.tokenizer.encode(source_text, return_tensors="pt", padding="max_length", truncation=True, max_length=17)
#         target_tokens = self.tokenizer.encode(target_text, return_tensors="pt", padding="max_length", truncation=True, max_length=20)
#         return source_tokens.squeeze(), target_tokens.squeeze()

    def __getitem__(self, idx):
        #####Remeber if its GPT the botht source and target max_length should be same######
        source_text = self.source_texts[idx]
        target_text = self.target_texts[idx]
        source_encoding = self.tokenizer(source_text,  
                                  return_tensors="pt", 
                                  padding="max_length", 
                                  truncation=True,
                                  max_length=17)
        target_encoding = self.tokenizer(target_text,
                                        return_tensors="pt", 
                                        padding="max_length", 
                                        truncation=True,
                                        max_length=21)
        return {
            "input_ids": source_encoding["input_ids"].squeeze(),
            "attention_mask": source_encoding["attention_mask"].squeeze(),
            "labels": target_encoding["input_ids"].squeeze()
        }



# Dataset tokenization and encoding

In [ ]:
# Define dataset and dataloader
dataset_train = TranslationDataset(X_train, y_train, tokenizer)
dataset_test = TranslationDataset(X_test, y_test, tokenizer)

In [ ]:
dataset_train[0]

# Dataloader

In [ ]:
dataloader_train = DataLoader(dataset_train, batch_size=2, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=2)

In [ ]:
for v in dataloader_train:
    print(v)
    break

# Model parameter

In [ ]:
import torch.nn as nn
num_epochs = 3
learning_rate = 3e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()


In [ ]:
#!pip3 install datasets

In [ ]:
import tqdm
from tqdm.notebook import tqdm

# Simple fine tune

In [ ]:
# Fine-tuning loop
model_type=model_name
for epoch in range(num_epochs):
    print("...Epoch..."+str(epoch+1))
    total_loss = 0.0
    for index ,batch in tqdm(enumerate(dataloader_train), total=len(dataloader_train), leave=True, disable=False):
        #print()
        #print("Epoch {} Batch {}/{}".format(epoch+1,index+1,len(dataloader_train)))  
        optimizer.zero_grad()
        input_ids = batch["input_ids"].squeeze().to(model.device)
        attention_mask = batch["attention_mask"].squeeze().to(model.device)
        labels = batch["labels"].squeeze().to(model.device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(dataloader_train)}")

In [ ]:
model.save_pretrained("saved_model/")
tokenizer.save_pretrained("saved_model/")

# Fine tuning with grad-accumulation and grad clipping

In [ ]:
import torch.nn as nn
num_epochs = 5
learning_rate = 5e-5
optimizer = AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
# Define the accumulation steps
accumulation_steps = 4
# text_column = "Tweet text"
# label_column = "text_label"

# Fine-tuning loop with gradient accumulation
for epoch in range(num_epochs):
    print("...Epoch..." ,str(epoch + 1))
    total_loss = 0.0
    optimizer.zero_grad()  # Clear gradients at the start of each epoch
    for index, batch in tqdm(enumerate(dataloader_train), total=len(dataloader_train), leave=True, disable=False):
        #print("Epoch {} Batch {}/{}".format(epoch + 1, index + 1, len(dataloader_train)))
        
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # Normalize the loss to account for gradient accumulation
        loss = loss / accumulation_steps
        loss.backward()
        
        # Step the optimizer after every accumulation step
        if (index + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps  # Multiply back to get the original loss
        
    # Final step for remaining gradients in the last mini-batch
    if (index + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(dataloader_train)}")
    print("....Model performance after epoch.."+str(epoch+1))
    

In [ ]:
y_test[1]

In [15]:
tokenizer = T5Tokenizer.from_pretrained('saved_model/')
model = T5ForConditionalGeneration.from_pretrained('saved_model/')

model.eval()  # Set to eval mode
#model.eval()
print("T5 prediction")
for i in range(10): 
    context = torch.tensor(tokenizer.encode(X_test[i]))
    context = torch.tensor(context, dtype=torch.long, device='cpu')
    context = context.unsqueeze(0)
    text = tokenizer.decode(model.generate(context,max_length=23)[0],
                            skip_special_tokens=True
                           )
    print("....english text............",X_test[i])
    print(".....original french text...",y_test[i])
    print(".....predicted text.........",text)
    print("**********************************************************************************************")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


T5 prediction


C:\Users\utsav\AppData\Local\Temp\ipykernel_22160\2119751610.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  context = torch.tensor(context, dtype=torch.long, device='cpu')


....english text............ our least favorite fruit is the banana , but your least favorite is the grape .
.....original french text... notre fruit pra©fa©ra© moins est la banane , mais votre moins pra©fa©ra© est le raisin .
.....predicted text......... notre fruit prafara moins est la banane, mais votre
**********************************************************************************************
....english text............ china is never mild during april , and it is sometimes wet in spring .
.....original french text... chine est jamais doux en avril , et il est parfois humide au printemps .
.....predicted text......... chine est jamais doux en avril, et il est parfois humide
**********************************************************************************************
....english text............ paris is sometimes rainy during winter , but it is chilly in fall .
.....original french text... paris est parfois pluvieux pendant l hiver , mais il est froid a l automne .
.....predic

In [ ]:
import onxx

In [ ]:
model.eval()
print("T5 prediction with grad accumulation step")
for i in range(10): 
    context = torch.tensor(tokenizer.encode(X_test[i]))
    context = torch.tensor(context, dtype=torch.long, device='cpu')
    context = context.unsqueeze(0)
    text = tokenizer.decode(model.generate(context,max_length=23)[0],skip_special_tokens=True)
    print("....english text............",X_test[i])
    print(".....original french text...",y_test[i])
    print(".....predicted text.........",text)
    print("..........................................................................................")

In [ ]:
###### BART RESULT ##########
model.eval()
print("RESULT WIH BART WIHTOUT GRAD ACCUMULATION")
for i in range(10): 
    context = torch.tensor(tokenizer.encode(X_test[i]))
    context = torch.tensor(context, dtype=torch.long, device='cpu')
    context = context.unsqueeze(0)
    text = tokenizer.decode(model.generate(context,max_length=512)[0],
                            skip_special_tokens=True
                           )
    print("....english text............",X_test[i])
    print(".....original french text...",y_test[i])
    print(".....predicted text.........",text)
    print("**********************************************************************************************")

In [ ]:
###### GPT RESULT ##########
# model.eval()
# for i in range(3): 
#     context = torch.tensor(tokenizer.encode(X_test[i]))
#     context = torch.tensor(context, dtype=torch.long, device='cpu')
#     context = context.unsqueeze(0)
#     text = tokenizer.decode(model.generate(context,max_length=512)[0],
#                             skip_special_tokens=True
#                            )
#     print("....english text............",X_test[i])
#     print(".....original french text...",y_test[i])
#     print(".....predicted text.........",text)
#     print("**********************************************************************************************")

In [ ]:
context = ["The", "cat"]
candidate_words = ["sat", "jumped", "slept"]
probabilities = [0.4, 0.3, 0.3]
k = 2  # Let's consider top 2 candidates for sampling

# Sort candidate words based on probabilities
sorted_candidates = [word for _, word in sorted(zip(probabilities, candidate_words), reverse=True)]
sorted_probabilities = sorted(probabilities, reverse=True)

# Select top-k candidates
top_k_candidates = sorted_candidates[:k]
top_k_probabilities = sorted_probabilities[:k]

# Normalize probabilities
normalized_probabilities = [prob / sum(top_k_probabilities) for prob in top_k_probabilities]

# Generate a random number between 0 and 1
random_number = 0.0223

# Sample from the normalized probabilities
cumulative_probability = 0
for index, prob in enumerate(normalized_probabilities):
    cumulative_probability += prob
    if random_number < cumulative_probability:
        selected_word = top_k_candidates[index]
        break

# Output the selected word
print("Selected Word:", selected_word)
